# Parte 2 — Consagração do Negócio: Entrega 1 — Modelagem Dimensional

Constrói o Star Schema com seis tabelas principais e três tabelas-ponte. As dimensões recebem chaves substitutas, e a fato reúne as métricas com uma linha por filme lançado. Ao final, são feitas checagens e uma consulta de teste da modelagem.


## Configuração da camada Gold

Usa o catálogo do projeto para definir os schemas Silver e Gold.


In [0]:
# Configura a origem Silver e o destino Gold da modelagem.

catalog = "cinedata_medallion"

bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")


DataFrame[]

## Apoio — Funções de checagem da modelagem

Define as funções usadas para verificar unicidade e preenchimento de chaves. Os resultados ficam registrados com o nome da tabela e da regra.


In [0]:
# Estas funções serão usadas nas verificações das tabelas Gold.

from datetime import datetime
from pyspark.sql import Row, functions as F

dq_results = []

def dq_check(table_name, check_name, df, condition):
    total = df.count()
    failed = df.filter(~F.coalesce(condition, F.lit(False))).count()
    passed = failed == 0

    dq_results.append(
        Row(
            tabela=table_name,
            verificacao=check_name,
            total_linhas=total,
            quantidade_falhas=failed,
            aprovado=passed,
            verificado_em=datetime.now()
        )
    )

    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")
    if not passed:
        raise ValueError(f"Falha de qualidade: {table_name} | {check_name}")
    return passed


def dq_check_unique(table_name, check_name, df, key_cols):
    total = df.count()
    duplicates = (
        df.groupBy(*key_cols)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    passed = duplicates == 0

    dq_results.append(
        Row(
            tabela=table_name,
            verificacao=check_name,
            total_linhas=total,
            quantidade_falhas=duplicates,
            aprovado=passed,
            verificado_em=datetime.now()
        )
    )

    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {duplicates} chaves duplicadas")
    if not passed:
        raise ValueError(f"Falha de qualidade: {table_name} | {check_name}")
    return passed


## Entrega 1 — Leitura das tabelas Silver

Carrega os metadados, dados financeiros, métricas, gêneros, entidades e avaliações que serão usados na modelagem dimensional.


In [0]:
# Lê as tabelas tratadas que alimentam as dimensões, a fato e as pontes.

df_info = spark.table(f"{silver_schema}.tb_info_filmes")
df_financeiro = spark.table(f"{silver_schema}.tb_financeiro_filmes")
df_metricas = spark.table(f"{silver_schema}.tb_metricas_engajamento")
df_generos = spark.table(f"{silver_schema}.tb_generos")
df_pessoas_empresas = spark.table(f"{silver_schema}.tb_pessoas_empresas")
df_avaliacoes = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
# A Gold não escolhe uma versão arbitrária para esconder duplicatas da Silver.
for nome, df in [("tb_info_filmes", df_info), ("tb_financeiro_filmes", df_financeiro), ("tb_metricas_engajamento", df_metricas)]:
    dq_check(f"silver.{nome}", "id_filme preenchido", df, F.col("id_filme").isNotNull())
    dq_check_unique(f"silver.{nome}", "um registro por filme", df, ["id_filme"])


[PASS] silver.tb_info_filmes | id_filme preenchido | 0/97879 linhas falharam
[PASS] silver.tb_info_filmes | um registro por filme | 0 chaves duplicadas
[PASS] silver.tb_financeiro_filmes | id_filme preenchido | 0/99006 linhas falharam
[PASS] silver.tb_financeiro_filmes | um registro por filme | 0 chaves duplicadas
[PASS] silver.tb_metricas_engajamento | id_filme preenchido | 0/99013 linhas falharam
[PASS] silver.tb_metricas_engajamento | um registro por filme | 0 chaves duplicadas


## Entrega 1 — gold.dim_movies

Cria uma linha por filme com os metadados solicitados. id_filme é a chave natural em STRING; sk_movie_id é a chave substituta BIGINT. A tabela é gravada e relida para que os joins usem as chaves salvas.


In [0]:
# A chave substituta identifica o filme nos relacionamentos da Gold.

df_dim_movies = (
    df_info
    .select(
        F.col("id_filme").cast("string").alias("id_filme"),
        F.col("titulo").cast("string").alias("titulo"),
        F.col("data_lancamento").cast("date").alias("data_lancamento"),
        F.col("ano_lancamento").cast("int").alias("ano_lancamento"),
        F.col("duracao_minutos").cast("int").alias("duracao_minutos"),
        F.col("idioma_original").cast("string").alias("idioma_original"),
        F.col("status_filme").cast("string").alias("status_filme"),
        F.col("sinopse").cast("string").alias("sinopse")
    )
    .withColumn("sk_movie_id", F.monotonically_increasing_id())
    .select(
        "sk_movie_id",
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse"
    )
)

df_dim_movies.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_movies")

# Usa as chaves gravadas na tabela
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")

display(df_dim_movies.limit(10))


sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
0,14564,Rings,2017-02-01,2017,102,en,Lançado,"\Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a \""\""movie within the movie\""\"" that no one has ever seen before.\"""""
1,38258,Grizzly II: Revenge,2020-02-17,2020,74,en,Lançado,\All hell breaks loose when a giant grizzly
2,38492,Billy Joel - Live at Yankee Stadium,2022-06-22,2022,86,en,Lançado,Billy Joel plays his greatest hits in the Big Apple.
3,42018,The Horse Thief,2019-03-19,2019,88,zh,Lançado,"Devout Buddhists, Norbu and Dolma live with their young son Tashi in a clan in Tibet. Norbu is a highwayman. After Norbu is charged with stealing from the temple, he and his family are banished. Impoverished and marginalized, they can do little when their beloved son becomes ill. Tashi dies of a fever. After a second son is born, Norbu focuses his every action on keeping this child alive, seeking re-admission to the clan for his wife and child, then risking all to save them from isolation and starvation in winter."
4,42330,Monkey Magic,2018-09-22,2018,66,zh,Lançado,"Dearth Voyd, the ruler of the dark side of the universe, wants to turn the human world into a place of violence and evil!! However, a huge meteor falls upon Flower-Fruit mountain giving birth to Kongo, a monkey made of stone! Kongo, now sets out on his quest to fight evil and restore order to a desperate land. Episodes 1-3"
5,43074,Ghostbusters,2016-07-14,2016,117,en,Lançado,"Following a ghost invasion of Manhattan, paranormal enthusiasts Erin Gilbert and Abby Yates, nuclear engineer Jillian Holtzmann, and subway worker Patty Tolan band together to stop the otherworldly threat."
6,47933,Independence Day: Resurgence,2016-06-22,2016,120,en,Lançado,"We always knew they were coming back. Using recovered alien technology, the nations of Earth have collaborated on an immense defense program to protect the planet. But nothing can prepare us for the aliens’ advanced and unprecedented force. Only the ingenuity of a few brave men and women can bring our world back from the brink of extinction."
7,50022,Super Troopers 2,2018-04-19,2018,100,en,Lançado,"When an international border dispute arises between the U.S. and Canada, the Super Troopers- Mac, Thorny, Foster, Rabbit and Farva, are called in to set up a new Highway Patrol station in the disputed area."
8,58431,Pride and Prejudice and Zombies,2016-02-04,2016,108,en,Lançado,"A zombie outbreak has fallen upon the land in this reimagining of Jane Austen's classic tale of the tangled relationships between lovers from different social classes in 19th century England. Feisty heroine Elizabeth Bennet is a master of martial arts and weaponry and the handsome Mr. Darcy is a fierce zombie killer, yet the epitome of upper class prejudice. As the zombie outbreak intensifies, they must swallow their pride and join forces on the blood-soaked battlefield in order to conquer the undead once and for all."
9,66213,Iruttu Araiyil Murattu Kuthu,2018-05-04,2018,118,ta,Lançado,A sexually-starved spirit traps two couples in a bungalow in Bangkok where they have planned to stay for a week. The spirit wants one of the two men to have sex with her. Can they manage to escape?


## Entrega 1 — gold.dim_genres

Cria o catálogo de gêneros distintos e gera sk_genre_id. A releitura da tabela permite usar as chaves gravadas na ponte de gêneros.


In [0]:
# Cada gênero distinto recebe sua chave substituta.

df_dim_genres = (
    df_generos
    .select(F.col("genero").alias("nome_genero"))
    .distinct()
    .withColumn("sk_genre_id", F.monotonically_increasing_id())
    .select("sk_genre_id", "nome_genero")
)

df_dim_genres.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_genres")

df_dim_genres = spark.table(f"{gold_schema}.dim_genres")

display(df_dim_genres.limit(10))

sk_genre_id,nome_genero
0,Horror
1,Western
2,Romance
3,Comedy
4,Family
5,Science Fiction
6,Documentary
7,Fantasy
8,Music
9,Thriller


## Entrega 1 — gold.dim_people

Consolida Ator, Diretor e Roteirista. A combinação de nome e tipo define cada registro, permitindo que uma pessoa tenha mais de uma atuação. Gera sk_person_id e grava a dimensão.


In [0]:
# Deduplica nome e tipo de atuação antes de gerar a chave da pessoa.

df_dim_people = (
    df_pessoas_empresas
    .filter(
        F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista")
    )
    .select(
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .distinct()
    .withColumn("sk_person_id", F.monotonically_increasing_id())
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

df_dim_people.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_people")

df_dim_people = spark.table(f"{gold_schema}.dim_people")

display(df_dim_people.limit(10))

sk_person_id,nome_pessoa,tipo_pessoa
0,Jeff Seibert,Ator
1,Hichem Mesbah,Ator
2,Joey Albright,Ator
3,Sylvester Mccoy,Ator
4,Elliot Daurat,Ator
5,Robert Patrick,Ator
6,Kasumi Arimura,Ator
7,Yuki Kaji,Ator
8,Elton John,Ator
9,Jalaja,Ator


## Entrega 1 — gold.dim_companies

Seleciona as entidades do tipo Produtora, cria um catálogo de nomes distintos e gera sk_company_id.


In [0]:
# Cada produtora distinta recebe uma chave usada na ponte de empresas.

df_dim_companies = (
    df_pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .distinct()
    .withColumn("sk_company_id", F.monotonically_increasing_id())
    .select("sk_company_id", "nome_produtora")
)

df_dim_companies.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_companies")

df_dim_companies = spark.table(f"{gold_schema}.dim_companies")

display(df_dim_companies.limit(10))

sk_company_id,nome_produtora
0,Vertigo Entertainment
1,Beijing Dadi Century
2,Mensch Productions
3,Brad Krevoy Television
4,Skywalk Films
5,Art Service
6,Colet & Co.
7,Oak Entertainments
8,Filmin' Tuscany
9,Tv Asahi


## Entrega 1 — gold.dim_reviews

Resume as avaliações por filme com contagem e média arredondada em duas casas. A contagem inclui todas as avaliações; a média ignora notas NULL. Associa o resumo à chave do filme e gera sk_review_id.


In [0]:
# As avaliações individuais são agrupadas em um resumo por filme.

df_avaliacoes_resumo = (
    df_avaliacoes
    .withColumn("id_filme", F.col("id_filme").cast("string"))
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(
            F.avg("nota_usuario"), 2
        ).cast("double").alias("nota_media_usuarios")
    )
)

df_dim_reviews = (
    df_avaliacoes_resumo
    .join(
        df_dim_movies.select("id_filme", "sk_movie_id"),
        on="id_filme",
        how="inner"
    )
    .withColumn("sk_review_id", F.monotonically_increasing_id())
    .select(
        "sk_review_id",
        "sk_movie_id",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios"
    )
)

df_dim_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_reviews")

df_dim_reviews = spark.table(f"{gold_schema}.dim_reviews")

display(df_dim_reviews.limit(10))

sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
0,1,2,5.15
1,2,1,2.5
2,8,1,4.7
3,9,1,4.9
4,15,1,1.2
5,17,1,1.2
6,20,2,8.1
7,21,1,7.0
8,22,1,5.3
9,24,1,4.0


## gold.fact_movies_performance

As tabelas Silver de informações, financeiro e métricas já têm uma linha por filme;
a leitura verifica essa condição antes dos joins. Usa filmes com status Lançado,
exclui datas futuras e mantém data ausente quando o status informa lançamento.
Os LEFT JOINs preservam filmes com métricas ausentes. Não escolhe versões arbitrárias.


In [0]:
# A unicidade da Silver foi validada antes de gravar as tabelas da Gold.

# Uma linha de dados financeiros por filme
df_financeiro_gold = (
    df_financeiro
    .select(
        F.col("id_filme").cast("string").alias("id_filme"),
        F.col("orcamento_usd").cast("decimal(18,2)").alias("orcamento_usd"),
        F.col("receita_usd").cast("decimal(18,2)").alias("receita_usd"),
        F.col("lucro_usd").cast("decimal(18,2)").alias("lucro_usd"),
        F.col("orcamento_brl").cast("decimal(18,2)").alias("orcamento_brl"),
        F.col("receita_brl").cast("decimal(18,2)").alias("receita_brl"),
        F.col("lucro_brl").cast("decimal(18,2)").alias("lucro_brl")
    )
)

# Uma linha de dados de engajamento por filme
df_metricas_gold = (
    df_metricas
    .select(
        F.col("id_filme").cast("string").alias("id_filme"),
        F.col("popularidade").cast("double").alias("popularidade"),
        F.col("nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
        F.expr("try_cast(qtd_votos_tmdb AS INT)").alias("qtd_votos_tmdb"),
        F.col("nota_media_imdb").cast("double").alias("nota_media_imdb"),
        F.expr("try_cast(qtd_votos_imdb AS INT)").alias("qtd_votos_imdb")
    )
)

# Consolida somente os filmes com status Lançado
df_fact_movies_performance = (
    df_dim_movies
    .filter(
    (F.col("status_filme") == "Lançado")
    & (
        F.col("data_lancamento").isNull()
        | (F.col("data_lancamento") <= F.current_date())
    )
)
    .select("id_filme", "sk_movie_id")

    .join(df_financeiro_gold, on="id_filme", how="left")
    .join(df_metricas_gold, on="id_filme", how="left")

    .select(
        "sk_movie_id",
        "orcamento_usd",
        "receita_usd",
        "lucro_usd",
        "orcamento_brl",
        "receita_brl",
        "lucro_brl",
        "popularidade",
        "nota_media_tmdb",
        "qtd_votos_tmdb",
        "nota_media_imdb",
        "qtd_votos_imdb"
    )
)

df_fact_movies_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fact_movies_performance")

display(df_fact_movies_performance.limit(10))


sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
0,25000000.00,83080890.00,58080890.00,124822500.00,414814575.68,289992075.68,24.584,4.966,2375,null,46286
1,7500000.00,null,null,37446750.00,null,null,null,3.161,28,null,null
2,null,null,null,null,null,null,2.98,7.1,null,7.8,212
3,null,null,null,null,null,null,3.403,6.63,27,6.8,1750
4,null,null,null,null,null,null,1.561,10.0,1,6.4,34
5,144000000.00,229147509.00,85147509.00,718977600.00,1144110597.69,425132997.69,40.052,5.371,5976,null,260417
6,165000000.00,389681935.00,224681935.00,823828500.00,1945642933.26,1121814433.26,31.224,5.283,5674,5.2,201715
7,null,18850674.00,null,null,94119530.21,null,12.698,5.854,417,null,31025
8,null,16457494.00,null,null,82170621.79,null,25.09,6.146,2029,null,65037
9,null,null,null,null,null,null,4.749,5.4,8,4.2,1019


## Entrega 1 — gold.bridge_movie_genre

Relaciona filmes e gêneros pelas chaves substitutas. Distinct garante um único par sk_movie_id e sk_genre_id para cada associação.


In [0]:
# A ponte representa os vários gêneros de um filme sem acrescentá-los à fato.

df_bridge_movie_genre = (
    df_generos
    .withColumn("id_filme", F.col("id_filme").cast("string"))

    .join(
        df_dim_movies.select("id_filme", "sk_movie_id"),
        on="id_filme",
        how="inner"
    )

    .join(
        df_dim_genres.withColumnRenamed("nome_genero", "genero"),
        on="genero",
        how="inner"
    )

    .select("sk_movie_id", "sk_genre_id")
    .distinct()
)

df_bridge_movie_genre.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_genre")

display(df_bridge_movie_genre.limit(10))

sk_movie_id,sk_genre_id
5519,10
77816,3
68102,16
28416,9
45075,11
71709,17
76356,0
46562,3
67524,0
63377,2


## Entrega 1 — gold.bridge_movie_person

Relaciona filmes às pessoas conforme nome e tipo de atuação. Conserva pares únicos de sk_movie_id e sk_person_id.


In [0]:
# A ponte permite relacionar vários atores, diretores e roteiristas ao mesmo filme.

df_bridge_movie_person = (
    df_pessoas_empresas
    .filter(
        F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista")
    )
    .withColumn("id_filme", F.col("id_filme").cast("string"))

    .join(
        df_dim_movies.select("id_filme", "sk_movie_id"),
        on="id_filme",
        how="inner"
    )

    .join(
        df_dim_people
        .withColumnRenamed("nome_pessoa", "nome_entidade")
        .withColumnRenamed("tipo_pessoa", "tipo_entidade"),
        on=["nome_entidade", "tipo_entidade"],
        how="inner"
    )

    .select("sk_movie_id", "sk_person_id")
    .distinct()
)

df_bridge_movie_person.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_person")

display(df_bridge_movie_person.limit(10))

sk_movie_id,sk_person_id
75409,192698
86622,144832
88779,338315
29224,289864
92282,289876
44320,289879
45765,247
18489,145079
97790,48510
22903,290034


## Entrega 1 — gold.bridge_movie_company

Relaciona filmes às produtoras e conserva pares únicos de sk_movie_id e sk_company_id.


In [0]:
# A ponte registra as produtoras associadas a cada filme.

df_bridge_movie_company = (
    df_pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .withColumn("id_filme", F.col("id_filme").cast("string"))

    .join(
        df_dim_movies.select("id_filme", "sk_movie_id"),
        on="id_filme",
        how="inner"
    )

    .join(
        df_dim_companies.withColumnRenamed(
            "nome_produtora", "nome_entidade"
        ),
        on="nome_entidade",
        how="inner"
    )

    .select("sk_movie_id", "sk_company_id")
    .distinct()
)

df_bridge_movie_company.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_company")

display(df_bridge_movie_company.limit(10))

sk_movie_id,sk_company_id
41,18354
49,35448
55,34804
143,10273
199,2597
252,38661
260,11024
311,42637
366,4712
395,7688


## Validação da Entrega 1 — Chaves, grão e esquemas

Confere chaves e associações únicas, preenchimento das chaves, unicidade dos filmes e dos resumos de avaliações e tipos de pessoa permitidos. printSchema exibe os tipos gravados para conferência.


In [0]:
# Os testes verificam o grão das tabelas e as regras das chaves.

tabelas_e_chaves = [
    ("dim_movies", ["sk_movie_id"]),
    ("dim_genres", ["sk_genre_id"]),
    ("dim_people", ["sk_person_id"]),
    ("dim_companies", ["sk_company_id"]),
    ("dim_reviews", ["sk_review_id"]),
    ("fact_movies_performance", ["sk_movie_id"]),
    ("bridge_movie_genre", ["sk_movie_id", "sk_genre_id"]),
    ("bridge_movie_person", ["sk_movie_id", "sk_person_id"]),
    ("bridge_movie_company", ["sk_movie_id", "sk_company_id"])
]

for tabela, chaves in tabelas_e_chaves:
    df_tabela = spark.table(f"{gold_schema}.{tabela}")

    dq_check_unique(
        f"gold.{tabela}",
        "chave ou associação única",
        df_tabela,
        chaves
    )

    for chave in chaves:
        dq_check(
            f"gold.{tabela}",
            f"{chave} não nula",
            df_tabela,
            F.col(chave).isNotNull()
        )

    df_tabela.printSchema()

dq_check_unique(
    "gold.dim_movies",
    "um registro por id_filme",
    spark.table(f"{gold_schema}.dim_movies"),
    ["id_filme"]
)

dq_check_unique(
    "gold.dim_reviews",
    "um resumo de avaliações por filme",
    spark.table(f"{gold_schema}.dim_reviews"),
    ["sk_movie_id"]
)

dq_check(
    "gold.dim_people",
    "tipo de pessoa permitido",
    spark.table(f"{gold_schema}.dim_people"),
    F.col("tipo_pessoa").isin("Ator", "Diretor", "Roteirista")
)

# Confere também as chaves naturais dos catálogos.
for tabela, chaves in [
    ("dim_genres", ["nome_genero"]),
    ("dim_people", ["nome_pessoa", "tipo_pessoa"]),
    ("dim_companies", ["nome_produtora"])
]:
    dq_check_unique(f"gold.{tabela}", "chave natural única", spark.table(f"{gold_schema}.{tabela}"), chaves)

# A fato precisa ter todos e somente os filmes que atendem ao recorte de lançamento.
df_lancados_esperados = df_dim_movies.filter(
    (F.col("status_filme") == "Lançado")
    & (F.col("data_lancamento").isNull() | (F.col("data_lancamento") <= F.current_date()))
)
if spark.table(f"{gold_schema}.fact_movies_performance").count() != df_lancados_esperados.count():
    raise ValueError("Quantidade de filmes na fato diferente do recorte esperado.")


[PASS] gold.dim_movies | chave ou associação única | 0 chaves duplicadas
[PASS] gold.dim_movies | sk_movie_id não nula | 0/97879 linhas falharam
root
 |-- sk_movie_id: long (nullable = true)
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- ano_lancamento: integer (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = true)
 |-- sinopse: string (nullable = true)

[PASS] gold.dim_genres | chave ou associação única | 0 chaves duplicadas
[PASS] gold.dim_genres | sk_genre_id não nula | 0/19 linhas falharam
root
 |-- sk_genre_id: long (nullable = true)
 |-- nome_genero: string (nullable = true)

[PASS] gold.dim_people | chave ou associação única | 0 chaves duplicadas
[PASS] gold.dim_people | sk_person_id não nula | 0/415819 linhas falharam
root
 |-- sk_person_id: long (nullable = true)
 |-- nome_pessoa: string (nullable

## Validação da Entrega 1 — FKs e consulta do Star Schema

Verifica as oito relações de chave estrangeira com as dimensões. Depois, junta fato e dimensão de filmes e exibe receita, lucro e quantidade de filmes por idioma e ano, demonstrando o consumo do Star Schema.


In [0]:
# A consulta final testa o uso conjunto da fato e da dimensão de filmes.

# Verifica se as chaves estrangeiras existem nas dimensões
relacoes = [
    ("fact_movies_performance", "dim_movies", "sk_movie_id"),
    ("dim_reviews", "dim_movies", "sk_movie_id"),
    ("bridge_movie_genre", "dim_movies", "sk_movie_id"),
    ("bridge_movie_genre", "dim_genres", "sk_genre_id"),
    ("bridge_movie_person", "dim_movies", "sk_movie_id"),
    ("bridge_movie_person", "dim_people", "sk_person_id"),
    ("bridge_movie_company", "dim_movies", "sk_movie_id"),
    ("bridge_movie_company", "dim_companies", "sk_company_id")
]

for tabela, dimensao, chave in relacoes:
    df_origem = spark.table(f"{gold_schema}.{tabela}")

    df_chaves = (
        spark.table(f"{gold_schema}.{dimensao}")
        .select(chave)
        .withColumn("_fk_existe", F.lit(True))
    )

    df_validacao = df_origem.join(
        df_chaves,
        on=chave,
        how="left"
    )

    dq_check(
        f"gold.{tabela}",
        f"{chave} existe em {dimensao}",
        df_validacao,
        F.col("_fk_existe")
    )
    
df_resultado = (
    spark.table(
        f"{gold_schema}.fact_movies_performance"
    ).alias("f")

    .join(
        spark.table(f"{gold_schema}.dim_movies").alias("m"),
        on="sk_movie_id",
        how="inner"
    )

    .groupBy("m.idioma_original", "m.ano_lancamento")

    .agg(
        F.round(
            F.sum("f.receita_brl"), 2
        ).alias("receita_total_brl"),

        F.round(
            F.sum("f.lucro_brl"), 2
        ).alias("lucro_total_brl"),

        F.count("*").alias("qtd_filmes")
    )

    .orderBy(F.desc("receita_total_brl"))
)

display(df_resultado)

[PASS] gold.fact_movies_performance | sk_movie_id existe em dim_movies | 0/96520 linhas falharam
[PASS] gold.dim_reviews | sk_movie_id existe em dim_movies | 0/27303 linhas falharam
[PASS] gold.bridge_movie_genre | sk_movie_id existe em dim_movies | 0/140437 linhas falharam
[PASS] gold.bridge_movie_genre | sk_genre_id existe em dim_genres | 0/140437 linhas falharam
[PASS] gold.bridge_movie_person | sk_movie_id existe em dim_movies | 0/754464 linhas falharam
[PASS] gold.bridge_movie_person | sk_person_id existe em dim_people | 0/754464 linhas falharam
[PASS] gold.bridge_movie_company | sk_movie_id existe em dim_movies | 0/114568 linhas falharam
[PASS] gold.bridge_movie_company | sk_company_id existe em dim_companies | 0/114568 linhas falharam


idioma_original,ano_lancamento,receita_total_brl,lucro_total_brl,qtd_filmes
en,2017,128269995579.82,90659472213.69,7132
en,2016,113278374099.40,69688134380.21,6587
en,2019,110874387504.67,79458710981.99,6855
en,2018,106116184486.16,73520569728.33,6801
en,2022,80205675269.47,48740996633.68,5631
en,2023,56386458269.95,32859112508.91,4245
en,2021,49024996335.61,23107646581.34,5921
zh,2023,16978208455.44,3207701916.07,154
en,2020,16202301974.36,2963117354.99,6181
zh,2021,16060838672.95,7743573164.76,245
